# Get temp data

This notebook uses PRISM weather data downloaded from University of Oregon.

https://prism.oregonstate.edu/downloads/

The Regression Ridge vineyard is large enough to be capture a single measurement in the PRISM model. These weather data can be a baseline for NDVI calculation

The below will read a bulk download of PRISM data, unzip the files, clip the relevant measurements while saving the results to a geo df, and then delete the extracted files to reduce storage ballooning.

In [1]:
import zipfile
import glob
from tqdm import tqdm
import os
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import pandas as pd
import shutil
vineyard = gpd.read_file('../data/polygons/RegressionRidge.geojson')


/home/simonhans/anaconda3/lib/python3.7/site-packages/geopandas/_compat.py:115: UserWarning: The Shapely GEOS version (3.11.4-CAPI-1.17.4) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  shapely_geos_version, geos_capi_version_string


In [2]:
# Full weather pull via gridMET — 2016-01-01 through 2025-12-31
# gridMET (IDAHO_EPSCOR/GRIDMET) is current through present, covers CONUS daily.
# Derives 6 PRISM-equivalent variables. Saves to ../data/PRISM/df.pkl

import ee
import pandas as pd
import geopandas as gpd

ee.Initialize(project='ee-simonhansedasi')

vineyard = gpd.read_file('../data/polygons/RegressionRidge.geojson')
vineyard_wgs = vineyard.to_crs('EPSG:4326')
geom = ee.Geometry(vineyard_wgs.geometry.unary_union.__geo_interface__)

gridmet = (ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
             .filterDate('2016-01-01', '2026-01-01')
             .select(['pr', 'tmmx', 'tmmn', 'vpd', 'rmax']))

print('gridMET images in range:', gridmet.size().getInfo())

def extract(img):
    # Derive PRISM-equivalent variables
    tmax  = img.select('tmmx').subtract(273.15)
    tmin  = img.select('tmmn').subtract(273.15)
    tmean = tmax.add(tmin).divide(2)

    # vpdmax: gridMET daily mean VPD (kPa) × 10 → hPa
    vpdmax = img.select('vpd').multiply(10)

    # vpdmin: es(tmin) × (1 - rmax/100) × 10 → hPa
    # es(T) = 0.6108 × exp(17.27T / (T + 237.3)) kPa  [Tetens]
    es_tmin = tmin.multiply(17.27).divide(tmin.add(237.3)).exp().multiply(0.6108)
    rmax    = img.select('rmax').divide(100)
    vpdmin  = es_tmin.multiply(ee.Image(1).subtract(rmax)).multiply(10)

    combined = (tmax.rename('tmax')
                    .addBands(tmin.rename('tmin'))
                    .addBands(tmean.rename('tmean'))
                    .addBands(img.select('pr').rename('ppt'))
                    .addBands(vpdmax.rename('vpdmax'))
                    .addBands(vpdmin.rename('vpdmin')))

    stats = combined.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom,
        scale=4000,
        bestEffort=True
    )
    return ee.Feature(None, stats).set('date', img.date().format('YYYY-MM-dd'))

print('Extracting... (may take 1-2 minutes)')
fc   = ee.FeatureCollection(gridmet.map(extract))
data = fc.getInfo()

rows = [f['properties'] for f in data['features']]
df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'Rows: {len(df)}  ({df["date"].min().date()} → {df["date"].max().date()})')
print(df[['date','ppt','tmax','tmean','tmin','vpdmax','vpdmin']].head())

out_path = '../data/PRISM/df.pkl'
df.to_pickle(out_path)
print(f'\nSaved to {out_path}')


/home/simonhans/anaconda3/lib/python3.7/site-packages/google/auth/crypt/_cryptography_rsa.py:22: CryptographyDeprecationWarning: Python 3.7 is no longer supported by the Python core team and support for it is deprecated in cryptography. The next release of cryptography will remove support for Python 3.7.
  import cryptography.exceptions


gridMET images in range: 3653
Extracting... (may take 1-2 minutes)
Rows: 3653  (2016-01-01 → 2025-12-31)
        date       ppt      tmax     tmean      tmin  vpdmax    vpdmin
0 2016-01-01  0.000000 -4.994952 -6.205034 -7.415116     1.1  0.440088
1 2016-01-02  0.000000 -5.805048 -7.305048 -8.805048     0.7  0.727885
2 2016-01-03  1.000000 -5.505030 -6.977509 -8.449988     0.0  0.644641
3 2016-01-04  0.000000 -3.160078 -4.837600 -6.515122     0.0  0.000000
4 2016-01-05  0.455039 -1.105036 -2.687591 -4.270146     0.0  0.000000

Saved to ../data/PRISM/df.pkl


In [3]:
# Find the updated PRISM collection and check coverage
import ee
ee.Initialize(project='ee-simonhansedasi')

# Try candidate IDs for the updated PRISM daily collection
candidates = [
    'OREGONSTATE/PRISM/AN81d',        # deprecated (ends 2020)
    'OREGONSTATE/PRISM/v2/AN81d',
    'OREGONSTATE/PRISM/AN91d',
    'OREGONSTATE/PRISM/Norm91m',
]

for cid in candidates:
    try:
        col = ee.ImageCollection(cid)
        n = col.size().getInfo()
        last = col.sort('system:time_start', False).first().date().format('YYYY-MM-dd').getInfo()
        print(f'  {cid}  n={n}  last={last}')
    except Exception as e:
        print(f'  {cid}  ERROR: {e}')

# Also check gridMET as fallback — covers CONUS daily through present
try:
    gm = ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
    n = gm.size().getInfo()
    last = gm.sort('system:time_start', False).first().date().format('YYYY-MM-dd').getInfo()
    print(f'  IDAHO_EPSCOR/GRIDMET  n={n}  last={last}')
    print('  gridMET bands:', gm.first().bandNames().getInfo())
except Exception as e:
    print(f'  gridMET ERROR: {e}')


/home/simonhans/anaconda3/lib/python3.7/site-packages/ee/deprecation.py:204: DeprecationWarning: 

Attention required for OREGONSTATE/PRISM/AN81d! You are using a deprecated asset.
To ensure continued functionality, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/OREGONSTATE_PRISM_AN81d

  warnings.warn(warning, category=DeprecationWarning)


  OREGONSTATE/PRISM/AN81d  n=14610  last=2020-12-30
  OREGONSTATE/PRISM/v2/AN81d  ERROR: ImageCollection.load: ImageCollection asset 'OREGONSTATE/PRISM/v2/AN81d' not found (does not exist or caller does not have access).
  OREGONSTATE/PRISM/AN91d  ERROR: ImageCollection.load: ImageCollection asset 'OREGONSTATE/PRISM/AN91d' not found (does not exist or caller does not have access).
  OREGONSTATE/PRISM/Norm91m  n=24  last=2020-12-01
  IDAHO_EPSCOR/GRIDMET  n=17266  last=2026-04-09
  gridMET bands: ['pr', 'rmax', 'rmin', 'sph', 'srad', 'th', 'tmmn', 'tmmx', 'vs', 'erc', 'eto', 'bi', 'fm100', 'fm1000', 'etr', 'vpd']


In [4]:
# set base weather directory
base_dir = 'data/PRISM'

'''
This will require the following file-structure

| data
| | PRISM
| | ppt
| | | 2015
| | | | file_1.zip
| | | | file_2.zip
| | | | ....
| | tmax
| | | 2015
| | | ...
| | | 2024
| | ...
| | vpdmin
'''

# weather features available from PRISM and relevant to NDVI
weather_features = ['ppt','tmax','tmean','tmin','vpdmax','vpdmin']
weather_folders = [
    os.path.join(base_dir, f) for f in weather_features
]

# Years of available PRISM data
years = range(2016, 2026, 1)

weather_folders_years = []
for folder in weather_folders:
    feature_years = [
        os.path.join(folder, str(year)) for year in years
    ]
    weather_folders_years.append(feature_years)

# wfy = weather feature years
# collapse weather_features_years into single list of data folders
weather_folders_years = [item for sublist in weather_folders_years for item in sublist]

In [5]:
import os, glob

# --- Diagnose what is actually on disk ---
print(f"base_dir resolves to: {os.path.abspath(base_dir)}")
print()

for feature in weather_features:
    for year in [2025]:
        folder = os.path.join(base_dir, feature, str(year))
        exists = os.path.isdir(folder)
        zips = glob.glob(os.path.join(folder, '*.zip')) if exists else []
        print(f"{feature}/2025  exists={exists}  zips={len(zips)}")


base_dir resolves to: /home/simonhans/coding/GeoGastronomy/GrapeExpectations/RegressionRidge/data_wrangling/data/PRISM

ppt/2025  exists=True  zips=0
tmax/2025  exists=True  zips=0
tmean/2025  exists=True  zips=0
tmin/2025  exists=True  zips=0
vpdmax/2025  exists=True  zips=0
vpdmin/2025  exists=True  zips=0


In [7]:
df

,date,ppt,tmax,tmean,tmin,vpdmax,vpdmin
0,2016-01-01,0.000000,-4.994952,-6.205034,-7.415116,1.100000,0.440088
1,2016-01-02,0.000000,-5.805048,-7.305048,-8.805048,0.700000,0.727885
2,2016-01-03,1.000000,-5.505030,-6.977509,-8.449988,0.000000,0.644641
3,2016-01-04,0.000000,-3.160078,-4.837600,-6.515122,0.000000,0.000000
4,2016-01-05,0.455039,-1.105036,-2.687591,-4.270146,0.000000,0.000000
...,...,...,...,...,...,...,...
3648,2025-12-27,0.000000,6.105038,2.199998,-1.705042,1.800000,0.066595
3649,2025-12-28,0.000000,4.950006,0.622482,-3.705042,1.755039,0.000000
3650,2025-12-29,0.000000,5.749994,0.949991,-3.850012,2.200000,0.112112
3651,2025-12-30,0.000000,4.249994,-0.322484,-4.894962,1.600000,0.000000


In [6]:
def unzip_file(zip_file_path, out_dir):
    with zipfile.ZipFile(zip_file_path, 'r') as z:
        first_member = z.namelist()[0]
        out_path = os.path.join(out_dir, first_member)
        if not os.path.exists(out_path):
            z.extractall(out_dir)
        extracted_files = [os.path.join(out_dir, f) for f in z.namelist()]
    return extracted_files


def clip_raster(file_path, weather_value, polygon=vineyard):
    with rasterio.open(file_path) as src:
        for idx, row in polygon.iterrows():
            geom = [row.geometry.__geo_interface__]
            out_image, out_transform = mask(src, geom, crop=True)
            data = out_image[0]
            nodata = src.nodata if src.nodata is not None else -9999
            vals = data[data != nodata]
            if len(vals) > 0:
                value = float(np.mean(vals))
                return {
                    'date': date,
                    'n': len(vals),
                    weather_value: value,
                }
            else:
                return None


def delete_extracted_files(file_list):
    for f in file_list:
        if os.path.exists(f):
            os.remove(f)
